# Convert SQLite outputs to parquet files with CytoTable

## Import libraries

In [1]:
import logging
import pathlib
import pprint

import pandas as pd
import pyarrow.parquet as pq

# cytotable will merge objects from SQLite file into single cells and save as parquet file
from cytotable import convert, presets

# Set the logging level to a higher level to avoid outputting unnecessary errors from config file in convert function
logging.getLogger().setLevel(logging.ERROR)


/media/18tbdrive/1.Github_Repositories/predicting_mouse_cardiac_fibrosis/environments/preprocessing_env/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Set paths and variables

In [2]:
# base directory where the validation plate's CellProfiler outputs are located
base_dir = pathlib.Path("../1.cellprofiler_processing/cp_output/").resolve(strict=True)

# find all plate folders that contain a CellProfiler SQLite output
plate_names = [
    p.name for p in base_dir.glob("CARD*") if p.is_dir() and list(p.glob("*.sqlite"))
]

# stop if no plates were found to avoid misleading success messages downstream
if not plate_names:
    raise FileNotFoundError(
        f"No plate folders with a CellProfiler SQLite file were found in {base_dir}"
    )

print(f"There are a total of {len(plate_names)} plate(s) to convert:")
for plate in plate_names:
    print(plate)


There are a total of 1 plate(s) to convert:
CARD-CelIns-CX7_261022150001


In [3]:
# preset configurations based on typical CellProfiler outputs
preset = "cellprofiler_sqlite_pycytominer"

# update preset to include site metadata and cell counts
joins = presets.config[preset]["CONFIG_JOINS"].replace(
    "Image_Metadata_Well,",
    "Image_Metadata_Well, Image_Metadata_Site, Image_Count_Cells,",
)

# Add the PathName columns separately
joins = joins.replace(
    "COLUMNS('Image_FileName_.*'),",
    "COLUMNS('Image_FileName_.*'),\n COLUMNS('Image_PathName_.*'),",
)

# type of file output from cytotable
dest_datatype = "parquet"

# directory for processed data
output_base = pathlib.Path("data")
converted_dir = output_base / "converted_profiles"
converted_dir.mkdir(parents=True, exist_ok=True)

# create plate info dictionary with paths needed for conversion
plate_info_dictionary = {
    name: {
        "path_to_sqlite": pathlib.Path(
            next(iter((base_dir / name).glob("*.sqlite")))
        ).resolve(strict=True),
        "path_to_output": converted_dir / f"{name}_converted.parquet",
    }
    for name in plate_names
}

# view the dictionary to assess that all info is added correctly
pprint.pprint(plate_info_dictionary, indent=4)


{   'CARD-CelIns-CX7_261022150001': {   'path_to_output': PosixPath('data/converted_profiles/CARD-CelIns-CX7_261022150001_converted.parquet'),
                                        'path_to_sqlite': PosixPath('/media/18tbdrive/1.Github_Repositories/predicting_mouse_cardiac_fibrosis/1.cellprofiler_processing/cp_output/CARD-CelIns-CX7_261022150001/cfret_features.sqlite')}}


## Convert SQLite to parquet files

In [4]:
for plate_name, info in plate_info_dictionary.items():
    # Skip plates that are already converted (CytoTable will not overwrite an existing file)
    if info["path_to_output"].is_file():
        print(f"Skipping conversion for plate {plate_name} as converted profile exists")
        continue

    print("Starting conversion with cytotable for plate:", plate_name)
    # Merge single cells and output as parquet file
    convert(
        source_path=str(info["path_to_sqlite"]),
        dest_path=str(info["path_to_output"]),
        dest_datatype=dest_datatype,
        preset=preset,
        joins=joins,
        chunk_size=5000,
    )

print("All plates have been converted with cytotable!")


Starting conversion with cytotable for plate: CARD-CelIns-CX7_261022150001
All plates have been converted with cytotable!


# Load in converted profiles to update

We will rename some of the columns (e.g., location centroids and cell count per FOV) to include Metadata prefix.

In [5]:
# List of columns to update with the "Metadata_" prefix
metadata_columns_to_update = [
    "Nuclei_Location_Center_X",
    "Nuclei_Location_Center_Y",
    "Cells_Location_Center_X",
    "Cells_Location_Center_Y",
    "Image_Count_Cells",
]

for plate_name, info in plate_info_dictionary.items():
    file_path = info["path_to_output"]
    if not file_path.is_file():
        print(f"Warning: file not found for plate {plate_name}")
        continue

    # Skip plates that already have the metadata updates (columns already prefixed)
    if "Metadata_Image_Count_Cells" in pq.read_schema(file_path).names:
        print(f"Skipping metadata updates for plate {plate_name} as already processed")
        continue

    # Load the DataFrame from the Parquet file
    df = pd.read_parquet(file_path)

    # Ensure Metadata_Plate contains only one unique value (occurs due to failure acquiring plate during run)
    if "Metadata_Plate" in df.columns and df["Metadata_Plate"].nunique() != 1:
        df["Metadata_Plate"] = plate_name

    # Drop rows where "Metadata_ImageNumber" is NaN
    df = df.dropna(subset=["Metadata_ImageNumber"])

    # Rearrange columns and add "Metadata_" prefix
    df = df[
        metadata_columns_to_update
        + [col for col in df.columns if col not in metadata_columns_to_update]
    ].rename(
        columns=lambda col: (
            "Metadata_" + col if col in metadata_columns_to_update else col
        )
    )

    # Save the processed DataFrame back to the same path
    df.to_parquet(file_path, index=False)
    print(f"Processed metadata columns for plate: {plate_name}")

print("All converted profiles have been updated with Metadata columns!")


Processed metadata columns for plate: CARD-CelIns-CX7_261022150001
All converted profiles have been updated with Metadata columns!


## Check output to confirm process worked

To confirm the number of single cells is correct, please use any database browser software to see if the number of rows in the "Per_Cells" compartment matches the number of rows in the data frame.

In [6]:
# pick a plate to inspect (there is currently only one validation plate)
first_plate = next(iter(plate_info_dictionary.keys()))
converted_path = plate_info_dictionary[first_plate]["path_to_output"]

# Load the selected converted parquet file
converted_df = pd.read_parquet(converted_path)

print(f"Loaded file: {converted_path}")
print(converted_df.shape)
converted_df.head()


Loaded file: data/converted_profiles/CARD-CelIns-CX7_261022150001_converted.parquet
(12259, 2486)


,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,Image_Metadata_Plate,Image_Metadata_Site,Image_Metadata_Well,Metadata_Cells_Number_Object_Number,...,Nuclei_Texture_Variance_ER_3_02_256,Nuclei_Texture_Variance_ER_3_03_256,Nuclei_Texture_Variance_Golgi_3_00_256,Nuclei_Texture_Variance_Golgi_3_01_256,Nuclei_Texture_Variance_Golgi_3_02_256,Nuclei_Texture_Variance_Golgi_3_03_256,Nuclei_Texture_Variance_Mito_3_00_256,Nuclei_Texture_Variance_Mito_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256,Nuclei_Texture_Variance_Mito_3_03_256
0,727.918281,600.988499,750.800987,589.914333,3,1197,CARD-CelIns-CX7_261022150001,21,G10,1,...,4.563536,4.480975,0.316509,0.319046,0.324920,0.308804,1.003260,0.966411,0.881937,0.894013
1,438.277217,618.968575,435.393650,626.874871,3,1178,CARD-CelIns-CX7_261022150001,2,G10,1,...,3.960367,3.530078,0.848414,0.839900,0.839055,0.836785,0.811888,0.769048,0.836701,0.805615
2,614.091837,116.685131,590.415144,124.465238,22,474,CARD-CelIns-CX7_261022150001,23,D05,1,...,4.233442,4.181577,5.857898,6.082198,5.968376,5.741251,4.612899,4.817645,5.004735,4.577183
3,608.491142,246.229175,578.388182,248.428514,8,1169,CARD-CelIns-CX7_261022150001,18,G09,1,...,1.235712,1.200017,0.677024,0.595237,0.611582,0.639537,1.123488,1.248175,1.118515,1.039515
4,840.720085,300.365919,786.055907,246.476835,7,534,CARD-CelIns-CX7_261022150001,8,D08,1,...,2.723099,2.949850,0.640618,0.620433,0.616328,0.617292,0.885188,0.830594,0.895156,1.010088
